# Lab 07-03: Connect Agents Across Teams With A2A

**Book chapter:** Chapter 07: Orchestrating Multi-Agent Systems  
**Estimated time:** 20 minutes

## Overview

Cymbal Retail's return pickups are scheduled by a parcel carrier that runs its own
agent. In this lab, you expose the carrier's ADK agent over the Agent2Agent (A2A)
protocol, read its Agent Card, and connect Cymbal Retail's returns coordinator to it
with RemoteA2aAgent.

## Learning Objectives

By the end of this lab, you will be able to:

- Expose an ADK agent as an A2A server with to_a2a
- Read and interpret an Agent Card
- Delegate to a remote agent with RemoteA2aAgent
- Explain when to choose A2A instead of MCP or sub-agents

## Before You Begin

- Complete **Lab 00: Set Up the Lab Environment** in the `Appendix-C-Setup` folder

**Cost:** A few cents for Gemini requests. No Google Cloud resources are created.

## How to Use This Notebook

Run the cells in order, from top to bottom, by selecting each code cell and pressing **Shift+Enter**. Read the explanation above each cell before you run it, and compare your output with the expected result that the text describes.

## Key Concepts

| Protocol or feature | Connects | Use when |
|---|---|---|
| **A2A** | Independent agents, often across teams or companies | The other agent must stay separate |
| **MCP** | An agent to tools and data | The agent needs a capability of its own |
| **Sub-agents and AgentTool** | Agents in the same application | One team owns all the agents |

## Setup

In [ ]:
%pip install --quiet --upgrade "google-adk[a2a]"

> **Note:** If you are using Google Colab and a message asks you to restart the
> session after the installation, click **Restart session**, and then continue
> from the next cell. You don't need to run the installation cell again.

In [ ]:
import os

# Replace with your lab project ID (see Appendix C).
PROJECT_ID = "your-project-id"  # @param {type:"string"}
# A current Gemini Flash model ID. Check the Google models page if this
# model has been retired.
MODEL = "gemini-3.8-flash"  # @param {type:"string"}

# Fall back to the environment variables set by labenv.sh (Appendix C).
if PROJECT_ID == "your-project-id":
    PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", PROJECT_ID)
MODEL = os.environ.get("GEMINI_MODEL", MODEL)
assert PROJECT_ID != "your-project-id", "Set PROJECT_ID to your project ID."
# The lab code reads these environment variables.
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GEMINI_MODEL"] = MODEL
print("Project:", PROJECT_ID)
print("Model:  ", MODEL)

In [ ]:
import sys

if "google.colab" in sys.modules:
    # In Colab, sign in with the Google account that owns the lab project.
    from google.colab import auth
    auth.authenticate_user(project_id=PROJECT_ID)
    print("Signed in to Google Cloud from Colab.")
else:
    # In Jupyter, VS Code, or Workbench, the client libraries use Application
    # Default Credentials. If a later cell reports a credentials error, run
    # this command in a terminal and then rerun the cell:
    #   gcloud auth application-default login
    print("Using Application Default Credentials.")

## Step 1: Create the carrier's agent

The carrier's agent has its own tool, `schedule_pickup`, which Cymbal Retail never sees directly.

In [ ]:
import json
import os
import threading
import time
import urllib.request

import uvicorn
from google.adk.a2a.utils.agent_to_a2a import to_a2a
from google.adk.agents import Agent
from google.adk.agents.remote_a2a_agent import AGENT_CARD_WELL_KNOWN_PATH
from google.adk.agents.remote_a2a_agent import RemoteA2aAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"
MODEL = os.environ.get("GEMINI_MODEL", "gemini-3.8-flash")
PORT = 8001


def schedule_pickup(return_id: str, postal_code: str) -> dict:
    """Schedules a parcel pickup for a return.

    Args:
        return_id: The return number, for example R-778.
        postal_code: The postal code for the pickup address.
    """
    return {"pickup_id": f"PU-{return_id[-3:]}", "postal_code": postal_code,
            "window": "tomorrow, 9:00 to 12:00"}


carrier_agent = Agent(
    name="carrier_agent", model=MODEL, tools=[schedule_pickup],
    description="Schedules parcel pickups for customer returns.",
    instruction="You work for the parcel carrier. Schedule pickups with "
                "schedule_pickup and confirm the pickup ID and window.")

## Step 2: Expose the agent over A2A and start the server

`to_a2a` wraps the agent as an A2A server application and generates its **Agent
Card**. The cell runs the application with uvicorn in a background thread. In
production, the carrier would run it in its own environment, such as Cloud Run or Agent
Runtime.

In [ ]:
a2a_app = to_a2a(carrier_agent, host="127.0.0.1", port=PORT)
server = uvicorn.Server(uvicorn.Config(a2a_app, host="127.0.0.1",
                                       port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
for attempt in range(50):
    if server.started:
        print(f"The carrier's A2A server is running on port {PORT}")
        break
    time.sleep(0.2)

**Expected result:** the server reports that it is running on port 8001.

## Step 3: Read the Agent Card that describes the remote agent

Remote agents publish an Agent Card at `/.well-known/agent-card.json`, which describes their name, purpose, and skills.

In [ ]:
CARD_URL = f"http://127.0.0.1:{PORT}{AGENT_CARD_WELL_KNOWN_PATH}"
card = json.load(urllib.request.urlopen(CARD_URL))
print("Agent Card from", CARD_URL)
print(json.dumps({key: card[key] for key in ("name", "description",
                                             "skills") if key in card},
                 indent=2)[:900])

**Expected result:** a card with the agent's name, description, and a skill for `schedule_pickup`.

## Step 4: Delegate to the remote agent from a local coordinator

`RemoteA2aAgent` reads the Agent Card and acts as a local stand-in for the remote agent,
so the coordinator can transfer to it like any other sub-agent.

In [ ]:
carrier = RemoteA2aAgent(
    name="carrier_agent", agent_card=CARD_URL,
    description="Schedules parcel pickups for returns with our carrier.")
coordinator = Agent(
    name="returns_coordinator", model=MODEL, sub_agents=[carrier],
    instruction="You coordinate Cymbal Retail returns. When the shopper "
                "needs a pickup, transfer to carrier_agent.")
runner = Runner(app_name="cymbal", agent=coordinator,
                auto_create_session=True,
                session_service=InMemorySessionService())
text = "Please schedule a pickup for return R-778 at postal code 80202."
message = types.Content(role="user", parts=[types.Part(text=text)])
print(f"\nUser: {text}")
answers = set()  # A remote agent can report the same answer twice.
for event in runner.run(user_id="shopper", session_id="s1",
                        new_message=message):
    for call in event.get_function_calls():
        print(f"  {event.author} calls {call.name}({dict(call.args)})")
    if event.is_final_response() and event.content and event.content.parts:
        answer = event.content.parts[0].text
        if answer not in answers:
            answers.add(answer)
            print(f"  Answer from {event.author}: {answer}")

**Expected result:** the coordinator transfers to `carrier_agent`, which schedules the pickup and returns a pickup ID and time window.

## Step 5: Stop the A2A server

Stop the A2A server.

In [ ]:
server.should_exit = True
time.sleep(1)
print("Stopped the carrier's A2A server")

## Check Your Understanding

**1. Where does the coordinator learn what the carrier agent can do?**

<details><summary>Show answer</summary>

From the Agent Card published at /.well-known/agent-card.json.

</details>

**2. Can the coordinator call schedule_pickup directly?**

<details><summary>Show answer</summary>

No. The tool belongs to the carrier. The coordinator can only send the carrier agent a request over A2A.

</details>

**3. Why choose A2A for the carrier instead of an MCP server?**

<details><summary>Show answer</summary>

The carrier is an independent organization with its own agent, so collaborating agent to agent keeps its tools, data, and logic private.

</details>

## Summary

You exposed an agent over A2A, read its Agent Card, and delegated to it from a coordinator
through RemoteA2aAgent. The server ran in this notebook and has been stopped, so there is
nothing to clean up.